# Data Cleaning Pipeline

This notebook applies a standardised cleaning pipeline to the raw car2go trip records
from 10 European cities. The cleaning logic is centralised in  to
guarantee that both modelling alternatives (XGBoost and LSTM) operate on identically
filtered data.

The pipeline addresses four classes of degenerate or anomalous records:
1. Trips with effectively zero displacement (distance < 100 m)
2. Trips with anomalous durations (< 1 min or > 8 h)
3. Trips outside the operational area (per-city geographic outliers at the 99.9th percentile)
4. Exact duplicate rows (re-ingestion artefacts)

The output is a single Parquet file () consumed by
all downstream notebooks.

## Configuration and Imports

In [ ]:
import pandas as pd
from pathlib import Path
from trip_cleaning import clean_trips

In [ ]:
DATA_DIR = Path("../datasets/cs_datasets")
OUTPUT_PATH = Path("../results/trips_cleaned.parquet")

CITIES = [
    "amsterdam", "berlin", "firenze", "kobenhavn",
    "milano", "muenchen", "roma", "stockholm", "torino", "wien",
]

print(f"Raw data directory: {DATA_DIR}")
print(f"Output path: {OUTPUT_PATH}")

## Loading and Cleaning All Cities

Each city's raw  file is loaded, cleaned with , and appended
to a common list. The function returns both the cleaned DataFrame and a per-step
audit dictionary for reporting.

In [ ]:
frames = []
audit_rows = []

for city in CITIES:
    path = DATA_DIR / f"{city}.txt"
    print(f"[{city}] Loading {path} ...")
    df = pd.read_csv(path, sep=" ", quotechar='"')
    df, stats = clean_trips(df, city)
    audit_rows.append(stats)
    df["city"] = city
    frames.append(df)

df_all = pd.concat(frames, ignore_index=True)
print(f"[{city}] Total cleaned trips: {len(df_all):,}")

## Cleaning Audit

The table below reports how many trips were removed by each filter, per city.
The overall discard rate is approximately 4.4%, dominated by the short-distance
filter (reservations where the vehicle did not actually move).

In [ ]:
df_audit = pd.DataFrame(audit_rows).set_index("city")
print(df_audit.to_string())

## Saving the Cleaned Dataset

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_all.to_parquet(OUTPUT_PATH, index=False)

print(f"Saved to {OUTPUT_PATH}")
print(f"  Rows:   {len(df_all):,}")
print(f"  Cities: {sorted(df_all['city'].unique())}")
print(f"  Size:   {OUTPUT_PATH.stat().st_size / 1024**2:.1f} MB")